# Phase 5: Data Parsers (GEO & SRA)

This notebook demonstrates `GEOParser` and `SRAParser` — given dataset accessions extracted from a paper, they fetch structured metadata and return `GEODataset` / `SRADataset` objects.

Cells 1–7 use **mocked HTTP / pysradb** (no network required).  Cells 8–9 are live demos gated on network access.

In [ ]:
import json
from unittest.mock import MagicMock, patch
import pandas as pd

from researcher_ai.models.dataset import (
    DataSource, GEODataset, SRADataset, SampleMetadata,
)
from researcher_ai.parsers.data.base import BaseDataParser
from researcher_ai.parsers.data.geo_parser import (
    GEOParser, _normalise_esummary, _df_to_samples as geo_df_to_samples,
)
from researcher_ai.parsers.data.sra_parser import (
    SRAParser, _df_to_samples as sra_df_to_samples,
    _unique_col, _first_col, _infer_experiment_type,
)

print('Imports OK')

## 1. Accession validation

`validate_accession()` is a pure regex check — no network call.  Both parsers implement `BaseDataParser` and reject each other's accession formats.

In [ ]:
geo = GEOParser()
sra = SRAParser()

geo_tests = [
    ('GSE72987',  True),
    ('GSM1000001', True),
    ('GPL570',    True),
    ('SRP062554', False),
    ('PRJNA123',  False),
]
sra_tests = [
    ('SRP062554', True),
    ('SRX1085399', True),
    ('SRR2071346', True),
    ('ERP001942',  True),
    ('DRR000001',  True),
    ('GSE72987',  False),
]

print('GEO validation:')
for acc, expected in geo_tests:
    result = geo.validate_accession(acc)
    mark = '✓' if result == expected else '✗ FAIL'
    print(f'  {mark}  {acc!r:18s} → {result}')

print('\nSRA validation:')
for acc, expected in sra_tests:
    result = sra.validate_accession(acc)
    mark = '✓' if result == expected else '✗ FAIL'
    print(f'  {mark}  {acc!r:18s} → {result}')

## 2. GEO esummary normalisation

`_normalise_esummary()` converts the raw NCBI E-utilities JSON record into a flat dict.  It handles semicolon-delimited taxon strings, numeric GPL IDs, and nested `extrelations` SubSeries links.

In [ ]:
raw_record = {
    'title': 'eCLIP-seq of HNRNPC in HEK293T',
    'summary': 'Enhanced CLIP to map HNRNPC binding sites genome-wide.',
    'gdstype': 'Expression profiling by high throughput sequencing',
    'entrytype': 'GSE',
    'taxon': 'Homo sapiens',
    'gpl': '11154',           # numeric; normaliser adds 'GPL' prefix
    'platformtitle': 'Illumina HiSeq 2000',
    'n_samples': '8',
    'extrelations': [],
    'accession': 'GSE72987',
    'uid': '200072987',
}

norm = _normalise_esummary(raw_record)
print('Normalised fields:')
for k, v in norm.items():
    print(f'  {k:20s} = {v!r}')

## 3. SuperSeries detection and child series extraction

A GEO SuperSeries is a GSE that bundles multiple child GSEs.  `_is_superseries()` reads the `gdstype` field; `_get_child_series()` extracts GSE IDs from the `relations` list.  For SuperSeries, `_fetch_samples()` is **not** called to avoid unbounded API requests.

In [ ]:
# Plain series
plain_md = _normalise_esummary(raw_record)
print('Is SuperSeries (plain):', geo._is_superseries(plain_md))

# SuperSeries
super_record = dict(raw_record, gdstype='Expression profiling by SuperSeries')
super_record['extrelations'] = [
    {'relationtype': 'SubSeries', 'targetobject': 'GSE72988'},
    {'relationtype': 'SubSeries', 'targetobject': 'GSE72989'},
]
super_md = _normalise_esummary(super_record)
print('Is SuperSeries (super):', geo._is_superseries(super_md))
children = geo._get_child_series('GSE72990', super_md)
print('Child series:', children)

## 4. GEOParser.parse() — mocked network

The full `parse()` call chains `_fetch_geo_metadata → _is_superseries → _fetch_samples → _fetch_processed_data`.  Here we mock both the metadata and the pysradb sample fetch.

In [ ]:
SAMPLE_DF = pd.DataFrame([{
    'run_accession': 'SRR2071346',
    'experiment_accession': 'SRX1085399',
    'sample_accession': 'SRS993095',
    'study_accession': 'SRP062554',
    'sample_title': 'HEK293T HNRNPC IP rep1',
    'scientific_name': 'Homo sapiens',
    'library_selection': 'IP',
    'library_layout': 'PAIRED',
    'instrument_model': 'Illumina HiSeq 2000',
    'library_source': 'TRANSCRIPTOMIC',
    'library_strategy': 'OTHER',
    'fastq_ftp': 'ftp.sra.ebi.ac.uk/vol1/fastq/SRR207/SRR2071346.fastq.gz',
}])

geo_mocked = GEOParser()
geo_mocked._fetch_geo_metadata = MagicMock(
    return_value=_normalise_esummary(raw_record)
)
geo_mocked._fetch_samples = MagicMock(
    return_value=geo_df_to_samples(SAMPLE_DF)
)

gds = geo_mocked.parse('GSE72987')

print(f'accession      : {gds.accession}')
print(f'title          : {gds.title}')
print(f'organism       : {gds.organism}')
print(f'series_type    : {gds.series_type}')
print(f'platform_id    : {gds.platform_id}')
print(f'total_samples  : {gds.total_samples}')
print(f'samples        : {len(gds.samples)}')
print(f'processed URLs : {gds.processed_data_urls}')
print(f'sample[0].id   : {gds.samples[0].sample_id}')
print(f'sample[0].fastq: {gds.samples[0].fastq_urls}')

## 5. SRA helper functions

`_unique_col`, `_first_col`, and `_infer_experiment_type` extract structured fields from the pysradb DataFrame.  They are tested here directly.

In [ ]:
SRA_DF = pd.DataFrame([
    {'run_accession': 'SRR2071346', 'experiment_accession': 'SRX1085399',
     'study_accession': 'SRP062554', 'scientific_name': 'Homo sapiens',
     'library_strategy': 'eCLIP', 'library_layout': 'PAIRED',
     'instrument_model': 'Illumina HiSeq 2000', 'sample_title': 'rep1',
     'library_source': 'TRANSCRIPTOMIC', 'library_selection': 'IP',
     'study_title': 'eCLIP of HNRNPC', 'fastq_ftp': ''},
    {'run_accession': 'SRR2071347', 'experiment_accession': 'SRX1085400',
     'study_accession': 'SRP062554', 'scientific_name': 'Homo sapiens',
     'library_strategy': 'eCLIP', 'library_layout': 'PAIRED',
     'instrument_model': 'Illumina HiSeq 2000', 'sample_title': 'rep2',
     'library_source': 'TRANSCRIPTOMIC', 'library_selection': 'IP',
     'study_title': 'eCLIP of HNRNPC', 'fastq_ftp': ''},
])

print('_unique_col(run_accession):', _unique_col(SRA_DF, 'run_accession'))
print('_first_col(study_title):   ', _first_col(SRA_DF, 'study_title'))
print('_infer_experiment_type:    ', _infer_experiment_type(SRA_DF))

## 6. SRAParser.parse() — mocked pysradb

The `parse()` dispatcher routes SRP → `_parse_project`, SRX → `_parse_experiment`, SRR → `_parse_run`.  Here we mock pysradb at the `_db` property level.

In [ ]:
sra_mocked = SRAParser()
mock_db = MagicMock()
mock_db.sra_metadata.return_value = SRA_DF
sra_mocked._db = mock_db

# Project-level
srp_ds = sra_mocked.parse('SRP062554')
print('=== SRP (project) ===')
print(f'  accession   : {srp_ds.accession}')
print(f'  srp         : {srp_ds.srp}')
print(f'  srx_list    : {srp_ds.srx_list}')
print(f'  srr_list    : {srp_ds.srr_list}')
print(f'  samples     : {len(srp_ds.samples)}')
print(f'  organism    : {srp_ds.organism}')

# Experiment-level
srx_ds = sra_mocked.parse('SRX1085399')
print('\n=== SRX (experiment) ===')
print(f'  accession   : {srx_ds.accession}')
print(f'  srx_list    : {srx_ds.srx_list}')

# Run-level
sra_mocked2 = SRAParser()
mock_db2 = MagicMock()
mock_db2.sra_metadata.return_value = SRA_DF.head(1)
sra_mocked2._db = mock_db2
srr_ds = sra_mocked2.parse('SRR2071346')
print('\n=== SRR (run) ===')
print(f'  accession   : {srr_ds.accession}')
print(f'  srr_list    : {srr_ds.srr_list}')
print(f'  total_samples: {srr_ds.total_samples}')

## 7. JSON round-trip verification

Both `GEODataset` and `SRADataset` must survive a Pydantic v2 `model_dump_json() → model_validate_json()` cycle without data loss.

In [ ]:
# GEO round-trip
gds_json = gds.model_dump_json(indent=2)
gds_restored = GEODataset.model_validate_json(gds_json)
assert gds_restored.accession == gds.accession
assert len(gds_restored.samples) == len(gds.samples)
print('GEODataset round-trip ✓')

# SRA round-trip
srp_json = srp_ds.model_dump_json(indent=2)
srp_restored = SRADataset.model_validate_json(srp_json)
assert srp_restored.accession == srp_ds.accession
assert srp_restored.srx_list == srp_ds.srx_list
print('SRADataset round-trip ✓')

print('\nSRADataset JSON preview (first 400 chars):')
print(srp_json[:400])

## 8. Live GEO parse (eCLIP paper GSE72987)

Requires network access.  Skipped if network is unavailable.  Fetches from NCBI E-utilities and bridges to SRA via pysradb.

In [ ]:
import socket
_has_network = True
try:
    socket.setdefaulttimeout(3)
    socket.socket(socket.AF_INET, socket.SOCK_STREAM).connect(('8.8.8.8', 53))
except Exception:
    _has_network = False

if not _has_network:
    print('No network — skipping live GEO demo.')
else:
    live_geo = GEOParser()
    gds_live = live_geo.parse('GSE72987')
    print(f'title         : {gds_live.title}')
    print(f'organism      : {gds_live.organism}')
    print(f'series_type   : {gds_live.series_type}')
    print(f'total_samples : {gds_live.total_samples}')
    print(f'platform_id   : {gds_live.platform_id}')
    print(f'processed URLs: {gds_live.processed_data_urls}')

## 9. Live SRA parse (SRP062554)

Requires network access.  Demonstrates project-level SRA parsing with pysradb for the same eCLIP study.

In [ ]:
if not _has_network:
    print('No network — skipping live SRA demo.')
else:
    live_sra = SRAParser()
    srp_live = live_sra.parse('SRP062554')
    print(f'accession     : {srp_live.accession}')
    print(f'organism      : {srp_live.organism}')
    print(f'experiment    : {srp_live.experiment_type}')
    print(f'total_samples : {srp_live.total_samples}')
    print(f'SRR count     : {len(srp_live.srr_list)}')
    if srp_live.samples:
        s = srp_live.samples[0]
        print(f'sample[0]     : {s.sample_id} | {s.title} | {s.layout}')

## Summary

Phase 5 delivers:

- **`BaseDataParser`** — abstract base class (`parse`, `validate_accession`) that all dataset-source parsers implement.
- **`GEOParser`** — parses GSE / GSM / GPL accessions via NCBI E-utilities.  Detects SuperSeries; bridges to SRA via pysradb for sample metadata.  Processed-data FTP URLs are constructed deterministically without a network call.
- **`SRAParser`** — parses SRP / SRX / SRR (and EBI/DDBJ equivalents) via pysradb.  Level is auto-detected from the accession prefix.  Network failures return stub datasets without raising.
- **JSON round-trip** — both `GEODataset` and `SRADataset` serialise and deserialise losslessly via Pydantic v2.
- **No live network required** for Cells 1–7; live demos in Cells 8–9 are auto-skipped when offline.